### BLEU, ROUGE, and BERTScore

Answering the underlying question behind every LLMs
- How to evaluate LLM output properly — and why each metric catches different things



In [1]:
%pip install rouge-score bert-score nltk -q


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00


In [24]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Understanding the metrics

#### BLEU (Bilingual Evaluation Understudy)
- built for machine translation in 2002
- Measures precision — what fraction of n-grams in thegenerated text
also appear in the reference text
- limitation :- It is precision only. Amodel that generates one correct word gets high BLEU-1 if that word is in the reference.
It does not care if the model missed important content entirely. IT works on short,exact answers where word choice matters

#### ROUGE (Recall-Oriented Understudy for Gisting Evaluation)
- built for summarization evaluation.It measures recall-what fraction of the reference text's n-grams
appeared in the generated text

- ROUGE-1: unigram overlap, ROUGE-2: bigram overlap
- weakness: purely lexical. "The dog bit the man" and
"The man bit the dog" get the same ROUGE score despite opposite meaning

#### BERTScore

- Uses contextual BERT embeddings to compare generated vs reference text.Matches each token in the generated text to the most similar token
in the reference text using cosine similarity

- Advantage :- returns Precision, Recall, and F1 at the semantic level
- Key weakness: slower, needs a model to run, less interpretable


In [18]:
## Building the testcases that will deliberately expose each metric's blind spot.

#Testcase 1 :Paraphrase (same meaning different words), Bleu and ROUGE will penalize it
reference_1 = "The capital of France is Paris."
generated_1 = "Paris serves as the capital city of France."

#TestCase 2 word overlap but wrong meaning
reference_2 = "The dog bit the man."
generated_2 = "The man bit the dog."

#TestCase 3  real rag chatbot output
reference_3 = """The Little Owl Golf Course was given its name because the narrator
and his friends decided to call it "Little Owl Golf Course."
It cost three cents a round, or two rounds for a nickel."""
generated_3 = """The Little Owl Golf Course was given its name because the narrator
and his friends decided to call it "Little Owl Golf Course."
It cost three cents a round or two for a nickel to play."""

#TestCase 4 partial overlapping
reference_4 = """The transformer architecture uses multi-head
self-attention, positional encodings, feed-forward layers, and
layer normalization to process sequential data in parallel."""
generated_4 = "Transformers use attention mechanisms to process data."

#TestCase 5 hallucination
reference_5 = "The Eiffel Tower is located in Paris."
generated_5 = "The Eiffel Tower is located in Paris and was built in 2015."

#Testcase 6 Irrelevant extra information
reference_6 = "Python is a programming language."
generated_6 = "Python is a programming language. Bananas are yellow. The Moon has no atmosphere."

#### Implementing BLEU

In [19]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

def compute_bleu(reference: str, generated: str) -> dict:
    ref_tokens = [word_tokenize(reference.lower())]
    gen_tokens = word_tokenize(generated.lower())
    smoothie = SmoothingFunction().method1
    return {
        "BLEU-1": round(sentence_bleu(ref_tokens, gen_tokens,
                        weights=(1,0,0,0), smoothing_function=smoothie), 4),
        "BLEU-2": round(sentence_bleu(ref_tokens, gen_tokens,
                        weights=(0.5,0.5,0,0), smoothing_function=smoothie), 4),
        "BLEU-4": round(sentence_bleu(ref_tokens, gen_tokens,
                        weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie), 4),
    }

#### Implementing ROUGE

In [27]:
def compute_rouge(reference: str, generated: str):

    scorer = rouge_scorer.RougeScorer(
        ["rouge1","rouge2","rougeL"],use_stemmer=True)
    scores = scorer.score(reference, generated)
    return {
        "ROUGE-1 P": round(scores["rouge1"].precision,4),
        "ROUGE-1 R": round(scores["rouge1"].recall,4),
        "ROUGE-1 F": round(scores["rouge1"].fmeasure,4),
        "ROUGE-2 F": round(scores["rouge2"].fmeasure,4),
        "ROUGE-L F": round(scores["rougeL"].fmeasure,4)
    }

#### Implementing BERTscore

In [21]:
def compute_bertscore(reference: str, generated: str):
    P,R,F1 = bert_score_fn(
        [generated],
        [reference],
        lang="en",
        verbose=False
    )

    return {
        "BERTScore P": round(P.item(),4),
        "BERTScore R": round(R.item(),4),
        "BERTScore F1": round(F1.item(),4)
    }

In [22]:
test_cases = [

    {
        "id":1,
        "name":"Paraphrase — same meaning, different words",
        "why":"BLEU/ROUGE should penalize. BERTScore should reward.",
        "reference":reference_1,
        "generated":generated_1
    },

    {
        "id":2,
        "name":"Wrong meaning — high word overlap",
        "why":"All metrics may score high. Exposes lexical blindness.",
        "reference":reference_2,
        "generated":generated_2
    },

    {
        "id":3,
        "name":"RAG chatbot output",
        "why":"Real chatbot response.",
        "reference":reference_3,
        "generated":generated_3
    },

    {
        "id":4,
        "name":"Partial coverage",
        "why":"Model missed important details.",
        "reference":reference_4,
        "generated":generated_4
    },

    {
        "id":5,
        "name":"Hallucination",
        "why":"Correct answer plus fabricated fact.",
        "reference":reference_5,
        "generated":generated_5
    },

    {
        "id":6,
        "name":"Irrelevant extra information",
        "why":"Correct answer with unnecessary text.",
        "reference":reference_6,
        "generated":generated_6
    }

]

In [25]:
rows = []
for tc in test_cases:
    bleu=compute_bleu(tc["reference"], tc["generated"])
    rouge=compute_rouge(tc["reference"], tc["generated"])
    bert=compute_bertscore(tc["reference"], tc["generated"])

    rows.append({

        "TC": tc["id"],
        "Test Case": tc["name"],

        "BLEU-1": bleu["BLEU-1"],
        "ROUGE-1 P": rouge["ROUGE-1 P"],
        "ROUGE-1 R": rouge["ROUGE-1 R"],
        "ROUGE-L F": rouge["ROUGE-L F"],
        "BERTScore F1": bert["BERTScore F1"],

        "Why it matters": tc["why"]

    })
results_df = pd.DataFrame(rows)
results_df

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,TC,Test Case,BLEU-1,ROUGE-1 P,ROUGE-1 R,ROUGE-L F,BERTScore F1,Why it matters
0,1,"Paraphrase — same meaning, different words",0.6667,0.6250,0.8333,0.5714,0.9494,BLEU/ROUGE should penalize. BERTScore should r...
1,2,Wrong meaning — high word overlap,1.0000,1.0000,1.0000,0.6000,0.9767,All metrics may score high. Exposes lexical bl...
2,3,RAG chatbot output,0.9500,0.9444,0.9714,0.9577,0.9885,Real chatbot response.
3,4,Partial coverage,0.0767,0.8571,0.2727,0.4138,0.8661,Model missed important details.
4,5,Hallucination,0.6154,0.5833,1.0000,0.7368,0.9664,Correct answer plus fabricated fact.
5,6,Irrelevant extra information,0.3750,0.3846,1.0000,0.5556,0.9244,Correct answer with unnecessary text.


#### Summary


In [26]:
summary = results_df[[
    "TC",
    "Test Case",
    "BLEU-1",
    "ROUGE-L F",
    "BERTScore F1"
]]

summary

,TC,Test Case,BLEU-1,ROUGE-L F,BERTScore F1
0,1,"Paraphrase — same meaning, different words",0.6667,0.5714,0.9494
1,2,Wrong meaning — high word overlap,1.0000,0.6000,0.9767
2,3,RAG chatbot output,0.9500,0.9577,0.9885
3,4,Partial coverage,0.0767,0.4138,0.8661
4,5,Hallucination,0.6154,0.7368,0.9664
5,6,Irrelevant extra information,0.3750,0.5556,0.9244
